Data Collection via OpenFDA API

In [1]:
import os
import json
import time
from datetime import datetime
from pathlib import Path

import requests
import polars as pl

## Data Paths

In [2]:
RAW_DATA_DIR = Path("../data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

INTERIM_DATA_DIR = Path("../data/interim")
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_DATA_DIR = Path("../data/processed")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

### API Key

In [3]:
API_KEY = os.getenv("OPENFDA_API_KEY")

if not API_KEY:
    raise ValueError("OPENFDA_API_KEY is not set.")

### Endpoint Config

In [4]:
BASE_URL = "https://api.fda.gov/drug/drugsfda.json"
SEARCH_QUERY = 'openfda.generic_name:*'
LIMIT = 99
SLEEP_SECONDS = 0.3
MAX_RECORDS = 50000

### Quick API Test

In [5]:
params = {
    "api_key": API_KEY,
    "search": SEARCH_QUERY,
    "limit": 1
}

response = requests.get(BASE_URL, params=params, timeout=30)
response.raise_for_status()

test_data = response.json()

print(test_data.keys())
print("Total matches:", test_data.get("meta", {}).get("results", {}).get("total"))

dict_keys(['meta', 'results'])
Total matches: 12344


### Helper Functions

In [6]:
def call_openfda(url, params, timeout=30, sleep_seconds=0.3):
    response = requests.get(url, params=params, timeout=timeout)
    response.raise_for_status()
    data = response.json()
    time.sleep(sleep_seconds)
    return data

def save_json_chunk(records, chunk_idx, output_dir):
    output_path = output_dir / f"openfda_raw_chunk_{chunk_idx:04d}.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2)
    return output_path

def fetch_openfda_in_chunks(
    base_url,
    api_key,
    search_query,
    output_dir,
    limit=99,
    max_records=50000,
    sleep_seconds=0.3
):
    all_saved_files = []
    all_records = []
    total_fetched = 0
    skip = 0
    chunk_idx = 1

    while total_fetched < max_records:
        params = {
            "api_key": api_key,
            "search": search_query,
            "limit": limit,
            "skip": skip
        }

        print(f"Requesting records {skip} to {skip + limit - 1}...")

        data = call_openfda(
            url=base_url,
            params=params,
            sleep_seconds=sleep_seconds
        )

        batch = data.get("results", [])
        if not batch:
            print("No more records returned. Stopping.")
            break

        remaining = max_records - total_fetched
        batch = batch[:remaining]

        saved_path = save_json_chunk(batch, chunk_idx, output_dir)
        all_saved_files.append(saved_path)

        all_records.extend(batch)

        batch_size = len(batch)
        total_fetched += batch_size

        print(f"Saved chunk {chunk_idx} with {batch_size} records.")
        print(f"Total fetched so far: {total_fetched}")

        if batch_size < limit:
            print("Last partial batch reached. Stopping.")
            break

        skip += limit
        chunk_idx += 1

    return all_saved_files, all_records, total_fetched

### Actual Collection

In [7]:
saved_files, all_records, total_fetched = fetch_openfda_in_chunks(
    base_url=BASE_URL,
    api_key=API_KEY,
    search_query=SEARCH_QUERY,
    output_dir=RAW_DATA_DIR,
    limit=LIMIT,
    max_records=MAX_RECORDS,
    sleep_seconds=SLEEP_SECONDS
)

print(f"\nFinished. Total records fetched: {total_fetched}")
print(f"Number of chunk files: {len(saved_files)}")

Requesting records 0 to 98...
Saved chunk 1 with 99 records.
Total fetched so far: 99
Requesting records 99 to 197...
Saved chunk 2 with 99 records.
Total fetched so far: 198
Requesting records 198 to 296...
Saved chunk 3 with 99 records.
Total fetched so far: 297
Requesting records 297 to 395...
Saved chunk 4 with 99 records.
Total fetched so far: 396
Requesting records 396 to 494...
Saved chunk 5 with 99 records.
Total fetched so far: 495
Requesting records 495 to 593...
Saved chunk 6 with 99 records.
Total fetched so far: 594
Requesting records 594 to 692...
Saved chunk 7 with 99 records.
Total fetched so far: 693
Requesting records 693 to 791...
Saved chunk 8 with 99 records.
Total fetched so far: 792
Requesting records 792 to 890...
Saved chunk 9 with 99 records.
Total fetched so far: 891
Requesting records 891 to 989...
Saved chunk 10 with 99 records.
Total fetched so far: 990
Requesting records 990 to 1088...
Saved chunk 11 with 99 records.
Total fetched so far: 1089
Requesting 

### Save Metadata Log

In [8]:
run_metadata = {
    "base_url": BASE_URL,
    "search_query": SEARCH_QUERY,
    "limit": LIMIT,
    "max_records_requested": MAX_RECORDS,
    "total_fetched": total_fetched,
    "num_chunk_files": len(saved_files),
    "chunk_files": [str(p) for p in saved_files],
    "timestamp": datetime.now().isoformat()
}

metadata_path = RAW_DATA_DIR / "openfda_collection_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, indent=2)

print(f"Saved metadata to {metadata_path}")

Saved metadata to ..\data\raw\openfda_collection_metadata.json


In [9]:
sample_file = saved_files[0]

with open(sample_file, "r", encoding="utf-8") as f:
    sample_records = json.load(f)

print(f"Loaded {len(sample_records)} records from {sample_file.name}")
print(sample_records[0].keys())

Loaded 99 records from openfda_raw_chunk_0001.json
dict_keys(['submissions', 'application_number', 'sponsor_name', 'openfda', 'products'])


### Save to CSV

In [10]:
raw_csv_rows = [
    {
        "record_index": i,
        "raw_json": json.dumps(record)
    }
    for i, record in enumerate(all_records)
]

raw_csv_df = pl.DataFrame(raw_csv_rows)

raw_csv_path = INTERIM_DATA_DIR / "openfda_raw_unprocessed.csv"
raw_csv_df.write_csv(raw_csv_path)

print(f"Saved raw unprocessed CSV to {raw_csv_path}")
print(raw_csv_df.head())

Saved raw unprocessed CSV to ..\data\interim\openfda_raw_unprocessed.csv
shape: (5, 2)
┌──────────────┬─────────────────────────────────┐
│ record_index ┆ raw_json                        │
│ ---          ┆ ---                             │
│ i64          ┆ str                             │
╞══════════════╪═════════════════════════════════╡
│ 0            ┆ {"submissions": [{"submission_… │
│ 1            ┆ {"submissions": [{"submission_… │
│ 2            ┆ {"submissions": [{"submission_… │
│ 3            ┆ {"submissions": [{"submission_… │
│ 4            ┆ {"submissions": [{"submission_… │
└──────────────┴─────────────────────────────────┘
